In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns
import glob
import ast
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
#X1Z, X2Z, X1W
# file path
path = "/scratch-cbe/users/sahana.narasimha/mcmc_run_ewkinos/logs/walker5*.log"
log_file = glob.glob(path)
log_file.remove("/scratch-cbe/users/sahana.narasimha/mcmc_run_ewkinos/logs/walker5.log")
log_file.append("/scratch-cbe/users/sahana.narasimha/mcmc_run_ewkinos/logs/walker60.log")
log_file = sorted(log_file)
log_file

In [ ]:
def read_log_file(file_path):
    
    step_list,muhat_list,proto_list,accratio_list = [],[],[],[]

    step_pattern = re.compile(r'\[randomWalker.*\] Step (\d+) begins')
    muhat_pattern = re.compile(r'with muhat ([\d.]+)')
    proto_pattern = re.compile(r"Protomodel: ({.*})")
    accratio_pattern = re.compile(r"Acceptance ratio (\d+(\.\d+)?)")

    none_pattern = re.compile(r'K is none, return to previous model') #need this when the second set of runs do not print the protomodel

    current_step = None
    current_muhat = np.nan
    current_proto = {}
    current_accratio = 0.0
    ctr = 0
    with open(file_path, 'r') as file:
        for line in file:
            step_match = step_pattern.search(line)
            if step_match:
                if current_step:
                    step_list.append(current_step)
                    muhat_list.append(current_muhat)
                    proto_list.append(current_proto)
                    accratio_list.append(current_accratio)
                
                current_step = int(step_match.group(1))
                #if current_step > 1400: print(current_step)
                current_muhat = np.nan
                current_proto = {}
                current_accratio = 0.0

            none_match = none_pattern.search(line)
            muhat_match = muhat_pattern.search(line)
            proto_match = proto_pattern.search(line)
            accratio_match = accratio_pattern.search(line)
            ctr = 0
            if muhat_match:
                ctr += 1
                try:
                    current_muhat = float(muhat_match.group(1))
                except ValueError:
                    current_muhat = np.nan

            if proto_match:
                ctr += 1
                current_proto = eval(proto_match.group(1))

            if accratio_match:
                ctr += 1
                current_accratio = float(accratio_match.group(1))
                            
            if none_match:
                if ctr != 3:            #if the protomodel is not printed because muhat did not converge to 1 in the second set of runs
                    current_muhat = None
                    current_proto = {}
                    current_accratio = None
            
        if current_step:
            step_list.append(current_step)
            muhat_list.append(current_muhat)
            proto_list.append(current_proto)
            accratio_list.append(current_accratio)
    
    return step_list, muhat_list, proto_list, accratio_list

In [ ]:
#Create pandas dataframe from log file
dataDict = {}
dataList = []
for i,file in enumerate(log_file):
    #if i==0:
    filename = file.split('/')[-1]
    dataDictFile = {}
    print(f"{filename}")
    step_list, muhat_list, proto_list,accratio_list = read_log_file(file)
    dataDictFile['filename'] = filename
    dataDictFile['X1Z mass'] = get_proto_par(proto_list[100:10000], 'mass', arg1=1000022)
    dataDictFile['X2Z mass'] = get_proto_par(proto_list[100:10000], 'mass', arg1=1000023)
    #dataDictFile['X3Z mass'] = get_proto_par(proto_list[100:10000], 'mass', arg1=1000025)
    dataDictFile['X1W mass'] = get_proto_par(proto_list[100:10000], 'mass', arg1=1000024)
    
    dataDictFile['SSM:(X1Z,X1Z)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000022, 1000022))
    dataDictFile['SSM:(X1Z,X2Z)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000022, 1000023))
    dataDictFile['SSM:(X1Z,X1W)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000022, 1000024))
    dataDictFile['SSM:(X1Z,-X1W)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(-1000024, 1000022))

    dataDictFile['SSM:(X2Z,X2Z)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000023, 1000023))
    dataDictFile['SSM:(-X1W,X1W)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(-1000024, 1000024))
    #dataDictFile['SSM:(X2Z,X3Z)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000023, 1000025))
    #dataDictFile['SSM:(X3Z,X3Z)'] = get_proto_par(proto_list[100:10000], 'ssm', arg1=(1000025, 1000025))
    '''
    dataDict['X2Z -> X1Z, h'] = get_proto_par(proto_list[100:3500], 'decays', arg1=1000023, arg2=(1000022, 25))
    dataDict['X1W -> X1Z, u,d'] = get_proto_par(proto_list[100:3500], 'decays', arg1=1000024, arg2=(1000022, 2, 1))
    dataDict['X1Z -> X1Z, l,nu'] = get_proto_par(proto_list[100:3500], 'decays', arg1=1000024, arg2=(1000022, 11, 12))
    '''
    print(len(dataDictFile['X1Z mass']))
    print(len(dataDictFile['X2Z mass']))
    print(len(dataDictFile['X1W mass']))
    #print(len(dataDictFile['X3Z mass']))

    if len(dataDictFile['X1Z mass']) > 1000: dataList.append(dataDictFile)
    #dataDict['nll'] = get_proto_par(proto_list[100:3500], 'nll')

mass1, mass2, mass3 = [],[],[]   
ssm1, ssm2, ssm3, ssm4, ssm5, ssm6 = [],[],[],[],[],[]     
for data in dataList:
    mass1 += data['X1Z mass']
    mass2 += data['X2Z mass']
    #mass3 += data['X3Z mass']
    mass3 += data['X1W mass']
    
    ssm1 += data['SSM:(X1Z,X1Z)']
    ssm2 += data['SSM:(X1Z,X2Z)']
    ssm3 += data['SSM:(X1Z,X1W)']
    ssm4 += data['SSM:(X1Z,-X1W)']
    ssm5 += data['SSM:(X2Z,X2Z)']
    ssm6 += data['SSM:(-X1W,X1W)']
    #ssm3 += data['SSM:(X2Z,X3Z)']
    #ssm4 += data['SSM:(X3Z,X3Z)']
    
dataDict['X1Z mass'] = mass1
dataDict['X2Z mass'] = mass2
#dataDict['X3Z mass'] = mass3
dataDict['X1W mass'] = mass3

dataDict['SSM:(X1Z,X1Z)'] = ssm1
dataDict['SSM:(X1Z,X2Z)'] = ssm2
dataDict['SSM:(X1Z,X1W)'] = ssm3
#dataDict['SSM:(X1Z,-X1W)'] = ssm4
dataDict['SSM:(X2Z,X2Z)'] = ssm5
dataDict['SSM:(-X1W,X1W)'] = ssm6


#dataDict['SSM:(X2Z,X3Z)'] = ssm3
#dataDict['SSM:(X3Z,X3Z)'] = ssm4

data = pd.DataFrame(dataDict)

In [ ]:
### Create corner plots
import corner
# Convert DataFrame to NumPy array
samples = data.values  # or data.to_numpy()

#Corner plot
figure = corner.corner(samples, labels=data.columns, quantiles=[0.16, 0.5, 0.84], 
                        range=[(10,1000), (10,1500), (10,2500), (1e-04, 50), (1e-04, 1e01), (1e-04,50), (1e-07,10),(1e-04,50)], 
                        axes_scale = ["linear"]*3 +["log"]*5,  show_titles=True)

plt.show()